# 1. Probability Basics

**Statistical Foundations for Data Science — Notebook 1 of 8**

Every machine learning model you will ever train is, underneath, a statement about
probability. A spam filter says *"given these words, the probability this email is spam
is 0.97"*. A weather model says *"70% chance of rain"*. A recommender says *"this user
will probably click this"*. Before you can reason about models, you need to reason about
uncertainty — and probability is the language we use for that.

### What you will learn

1. The vocabulary: experiment, sample space, event, outcome
2. Three ways people define probability (classical, empirical, subjective)
3. The three axioms every probability must obey
4. Rules for combining events: addition, complement, multiplication
5. Conditional probability and independence
6. Law of total probability and **Bayes' theorem**
7. Counting techniques: permutations and combinations
8. Simulation as a tool for checking your maths

### How to use this notebook

Run every cell. When you see a **Try it yourself** box, actually edit the code and
re-run it — probability is deeply unintuitive, and the fastest way to build intuition is
to watch simulations disagree with your gut.

In [ ]:
# Standard imports used throughout this notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product, permutations, combinations
from math import comb, perm, factorial

# A fixed seed makes every random result in this notebook reproducible.
# Change the number if you want to see a different random world.
rng = np.random.default_rng(seed=42)

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
print("NumPy version:", np.__version__)

---
## 1.1 The vocabulary of probability

Three words do most of the work. Learn them precisely, because sloppy vocabulary is the
number-one cause of wrong probability answers.

| Term | Meaning | Example (rolling one die) |
|---|---|---|
| **Random experiment** | A process whose result cannot be predicted with certainty, but which can be repeated | Rolling a fair six-sided die |
| **Outcome** | One single possible result | Rolling a `4` |
| **Sample space** $S$ | The set of *all* possible outcomes | $S = \{1,2,3,4,5,6\}$ |
| **Event** $E$ | Any **subset** of the sample space — a question you can ask that is either true or false after the experiment | "the roll is even" $= \{2,4,6\}$ |

Key idea: **an event is a set**. That is why set operations (union, intersection,
complement) become probability rules. Everything in this notebook follows from that.

In [ ]:
# Sample space of one die roll
S_die = {1, 2, 3, 4, 5, 6}

# Events are just subsets (Python sets are perfect for this)
even     = {x for x in S_die if x % 2 == 0}
gt_four  = {x for x in S_die if x > 4}

print("Sample space S      :", sorted(S_die))
print("Event 'even'        :", sorted(even))
print("Event 'greater than 4':", sorted(gt_four))

# Set operations = event operations
print("\neven OR  gt_four (union)       :", sorted(even | gt_four))
print("even AND gt_four (intersection):", sorted(even & gt_four))
print("NOT even         (complement)  :", sorted(S_die - even))

### Sample spaces get big, fast

If you roll **two** dice, the sample space is every ordered pair — $6 \times 6 = 36$
outcomes. `itertools.product` builds these for us. Notice that the outcomes are equally
likely, but the *sums* are not: there is one way to get a sum of 2 and six ways to get a
sum of 7. This is the single most important lesson in elementary probability —
**count the outcomes, not the labels you care about.**

In [ ]:
S_two_dice = list(product(range(1, 7), repeat=2))
print("Number of outcomes:", len(S_two_dice))
print("First 6 outcomes  :", S_two_dice[:6])

# How many outcomes give each sum?
sums = pd.Series([a + b for a, b in S_two_dice])
counts = sums.value_counts().sort_index()

table = pd.DataFrame({
    "sum": counts.index,
    "favourable_outcomes": counts.values,
    "probability": counts.values / len(S_two_dice),
})
table["probability"] = table["probability"].round(4)
table

In [ ]:
plt.bar(table["sum"], table["probability"], color="steelblue", edgecolor="black")
plt.xlabel("Sum of two dice")
plt.ylabel("Probability")
plt.title("Not all sums are equally likely (7 is the most common)")
plt.xticks(range(2, 13))
plt.show()

---
## 1.2 Three ways to define probability

**1. Classical (theoretical).** When every outcome is equally likely:

$$P(E) = \frac{\text{number of outcomes in } E}{\text{number of outcomes in } S}$$

Works for dice, cards, coins — anything with perfect symmetry.

**2. Empirical (frequentist).** Repeat the experiment $n$ times and count:

$$P(E) \approx \frac{\text{number of times } E \text{ happened}}{n}$$

This is what you use for real data ("what fraction of our customers churned?").

**3. Subjective (Bayesian).** A degree of belief — *"I'm 80% sure this deployment will
break something"*. There is no repeatable experiment, but the number still has to obey
the same rules. This view underlies Bayesian machine learning.

In [ ]:
# Classical probability: P(sum == 7) computed by counting
favourable = [o for o in S_two_dice if sum(o) == 7]
p_classical = len(favourable) / len(S_two_dice)
print(f"Classical  P(sum = 7) = {len(favourable)}/{len(S_two_dice)} = {p_classical:.4f}")

# Empirical probability: simulate 100,000 rolls of two dice and count
n = 100_000
rolls = rng.integers(1, 7, size=(n, 2))       # each row is one experiment
p_empirical = (rolls.sum(axis=1) == 7).mean()
print(f"Empirical  P(sum = 7) = {p_empirical:.4f}   (from {n:,} simulated rolls)")
print(f"Difference           = {abs(p_classical - p_empirical):.4f}")

### The Law of Large Numbers

Why did the simulation land so close to the truth? Because of the **Law of Large
Numbers**: as the number of independent repetitions grows, the empirical proportion
converges to the true probability.

The plot below shows the running estimate of $P(\text{sum}=7)$ after each roll. It swings
wildly at first, then settles. **Small samples lie.** This is exactly why a model
evaluated on 20 test rows tells you almost nothing.

In [ ]:
is_seven = (rolls.sum(axis=1) == 7).astype(int)
running = np.cumsum(is_seven) / np.arange(1, n + 1)

plt.plot(running, lw=1, color="steelblue", label="running empirical estimate")
plt.axhline(p_classical, color="crimson", ls="--", lw=2, label=f"true value = {p_classical:.4f}")
plt.xscale("log")
plt.xlabel("Number of rolls (log scale)")
plt.ylabel("Estimated P(sum = 7)")
plt.title("Law of Large Numbers in action")
plt.legend()
plt.show()

---
## 1.3 The three axioms

Every valid probability assignment must satisfy these three rules (Kolmogorov, 1933).
Everything else in probability is *derived* from them.

1. **Non-negativity:** $P(E) \ge 0$ for every event $E$
2. **Normalisation:** $P(S) = 1$ — something must happen
3. **Additivity:** if $A$ and $B$ are **mutually exclusive** (cannot both happen, $A \cap B = \emptyset$), then $P(A \cup B) = P(A) + P(B)$

Two immediate consequences you will use constantly:

- **Complement rule:** $P(E^c) = 1 - P(E)$
- **Range:** $0 \le P(E) \le 1$. If you ever compute a probability of 1.4, you have a bug.

### The general addition rule

If the events *can* overlap, adding them double-counts the overlap, so we subtract it:

$$P(A \cup B) = P(A) + P(B) - P(A \cap B)$$

In [ ]:
# Verify the addition rule on the two-dice sample space
A = [o for o in S_two_dice if o[0] == 6]          # first die is a 6
B = [o for o in S_two_dice if sum(o) >= 10]        # sum is at least 10

setA, setB = set(A), set(B)
N = len(S_two_dice)

pA        = len(setA) / N
pB        = len(setB) / N
pA_and_B  = len(setA & setB) / N
pA_or_B   = len(setA | setB) / N

print(f"P(A)          = {pA:.4f}   (first die is 6)")
print(f"P(B)          = {pB:.4f}   (sum >= 10)")
print(f"P(A and B)    = {pA_and_B:.4f}")
print(f"P(A or B)     = {pA_or_B:.4f}   <- counted directly")
print(f"P(A)+P(B)-P(A and B) = {pA + pB - pA_and_B:.4f}   <- addition rule")

# Naive (wrong) answer if you forget to subtract the overlap:
print(f"\nWrong answer without subtracting overlap: {pA + pB:.4f}")

---
## 1.4 Conditional probability

**Conditional probability** answers: *given that $B$ already happened, how likely is $A$?*

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}, \qquad P(B) > 0$$

Intuition: learning that $B$ happened **shrinks the sample space** down to $B$. You then
ask what fraction of that smaller world also has $A$ in it.

> **Read the notation carefully.** $P(A \mid B)$ and $P(B \mid A)$ are different numbers.
> Confusing them is called the *prosecutor's fallacy* and it has put innocent people in
> prison. We come back to this in the Bayes section.

In [ ]:
# Question: a die is rolled and you are told the result is even.
# What is the probability it is greater than 3?

B = {2, 4, 6}          # the conditioning event (what we were told)
A = {4, 5, 6}          # the event we care about

p_B      = len(B) / 6
p_A_and_B = len(A & B) / 6
p_A_given_B = p_A_and_B / p_B

print(f"P(B)          = {p_B:.4f}")
print(f"P(A and B)    = {p_A_and_B:.4f}")
print(f"P(A | B)      = {p_A_given_B:.4f}")
print(f"Unconditional P(A) = {len(A)/6:.4f}   <- knowing B changed our answer")

In [ ]:
# The same idea on a realistic dataset: does having a loyalty card relate to churn?
data = pd.DataFrame({
    "loyalty_card": ["yes"] * 600 + ["no"] * 400,
    "churned":      ["yes"] * 90 + ["no"] * 510 + ["yes"] * 160 + ["no"] * 240,
})

ct = pd.crosstab(data["loyalty_card"], data["churned"], margins=True, margins_name="Total")
print("Contingency table (counts):")
print(ct, "\n")

# P(churn) overall  vs  P(churn | loyalty card)
p_churn                = (data["churned"] == "yes").mean()
p_churn_given_card     = (data.loc[data.loyalty_card == "yes", "churned"] == "yes").mean()
p_churn_given_no_card  = (data.loc[data.loyalty_card == "no",  "churned"] == "yes").mean()

print(f"P(churn)                    = {p_churn:.3f}")
print(f"P(churn | has loyalty card) = {p_churn_given_card:.3f}")
print(f"P(churn | no loyalty card)  = {p_churn_given_no_card:.3f}")

---
## 1.5 Independence

Two events are **independent** when knowing one tells you nothing about the other:

$$P(A \mid B) = P(A) \quad\Longleftrightarrow\quad P(A \cap B) = P(A)\,P(B)$$

The second form is the usual test because it is symmetric and does not blow up when
$P(B)=0$.

**Multiplication rule (general):** $P(A \cap B) = P(A \mid B)\, P(B)$
**Multiplication rule (independent):** $P(A \cap B) = P(A)\,P(B)$

⚠️ **Independent ≠ mutually exclusive.** Mutually exclusive events are strongly
*dependent*: if $A$ happened, you know for certain $B$ did not.

In [ ]:
def is_independent(setA, setB, universe, tol=1e-12):
    '''Check P(A and B) == P(A) * P(B) on a finite equally-likely sample space.'''
    N = len(universe)
    pA, pB = len(setA) / N, len(setB) / N
    p_joint = len(setA & setB) / N
    return abs(p_joint - pA * pB) < tol, pA, pB, p_joint

U = set(S_two_dice)

pair1 = ({o for o in U if o[0] % 2 == 0},        # first die even
         {o for o in U if o[1] == 5})            # second die is 5
pair2 = ({o for o in U if o[0] % 2 == 0},        # first die even
         {o for o in U if sum(o) == 7})          # sum is 7
pair3 = ({o for o in U if o[0] == 1},            # first die is 1
         {o for o in U if sum(o) >= 10})         # sum >= 10

for name, (X, Y) in [("first even  &  second is 5", pair1),
                     ("first even  &  sum is 7",    pair2),
                     ("first is 1  &  sum >= 10",   pair3)]:
    indep, pX, pY, pXY = is_independent(X, Y, U)
    verdict = "INDEPENDENT" if indep else "DEPENDENT"
    print(f"{name:28s}  P(X)={pX:.3f}  P(Y)={pY:.3f}  P(X,Y)={pXY:.3f}  "
          f"P(X)P(Y)={pX*pY:.3f}  -> {verdict}")

The third pair is the interesting one: if the first die is a 1, the sum can never reach
10, so $P(X \cap Y) = 0$. The events are **mutually exclusive**, and precisely for that
reason they are **not independent**.

### Independence chains: the "at least one" trick

Many questions of the form *"what is the probability of at least one …"* are painful to
compute directly but easy via the complement:

$$P(\text{at least one}) = 1 - P(\text{none})$$

In [ ]:
# A server request fails independently with probability 0.02.
# What is the probability that at least one of 50 requests fails?
p_fail = 0.02
k = 50

p_none = (1 - p_fail) ** k
print(f"P(no failures in {k})        = {p_none:.4f}")
print(f"P(at least one failure)    = {1 - p_none:.4f}")

# Simulation check
trials = rng.random((200_000, k)) < p_fail
print(f"Simulated                  = {trials.any(axis=1).mean():.4f}")

---
## 1.6 Law of Total Probability

Sometimes you cannot get $P(A)$ directly, but you can get it *within each group*. If
$B_1, B_2, \dots, B_k$ **partition** the sample space (mutually exclusive, and together
cover everything), then:

$$P(A) = \sum_{i=1}^{k} P(A \mid B_i)\, P(B_i)$$

In words: the overall rate is the **weighted average** of the group rates, weighted by
group size. This is the formula behind Simpson's paradox, class-imbalance corrections,
and mixture models.

In [ ]:
# A factory has three machines producing the same part.
machines = pd.DataFrame({
    "machine":       ["A",    "B",    "C"],
    "share_of_output": [0.50,  0.30,   0.20],   # P(B_i)
    "defect_rate":     [0.01,  0.03,   0.06],   # P(defect | B_i)
})

machines["contribution"] = machines["share_of_output"] * machines["defect_rate"]
p_defect = machines["contribution"].sum()

print(machines.to_string(index=False))
print(f"\nP(defect) = {p_defect:.4f}  ({p_defect*100:.2f}% of all parts)")

---
## 1.7 Bayes' theorem

Bayes' theorem lets you **flip** a conditional probability around:

$$P(B \mid A) = \frac{P(A \mid B)\,P(B)}{P(A)}
             = \frac{P(A \mid B)\,P(B)}{\sum_i P(A \mid B_i)\,P(B_i)}$$

The vocabulary:

- $P(B)$ — the **prior**: what you believed before seeing evidence
- $P(A \mid B)$ — the **likelihood**: how well the hypothesis explains the evidence
- $P(B \mid A)$ — the **posterior**: your updated belief
- $P(A)$ — the **evidence** / normalising constant

### The classic worked example (and the classic trap)

A disease affects **1 in 1000** people. A test is **99% sensitive** (detects the disease
when present) and **95% specific** (correctly clears healthy people). You test positive.

Most people guess "about 99%". Let's compute it.

In [ ]:
prior_disease = 0.001          # P(D)
sensitivity   = 0.99           # P(+ | D)
specificity   = 0.95           # P(- | not D)  ->  P(+ | not D) = 0.05

p_pos_given_D    = sensitivity
p_pos_given_notD = 1 - specificity

# Law of total probability for the denominator
p_pos = p_pos_given_D * prior_disease + p_pos_given_notD * (1 - prior_disease)

posterior = (p_pos_given_D * prior_disease) / p_pos

print(f"P(positive test)        = {p_pos:.5f}")
print(f"P(disease | positive)   = {posterior:.5f}  ->  about {posterior*100:.1f}%")

In [ ]:
# Why so low? Think in raw counts out of 100,000 people.
N = 100_000
sick     = N * prior_disease
healthy  = N - sick

true_pos  = sick * sensitivity
false_pos = healthy * (1 - specificity)

print(f"Out of {N:,} people:")
print(f"  {sick:,.0f} are sick     -> {true_pos:,.0f} test positive (true positives)")
print(f"  {healthy:,.0f} are healthy -> {false_pos:,.0f} test positive (FALSE positives)")
print(f"\nSo among {true_pos + false_pos:,.0f} positive results, only {true_pos:,.0f} are real:")
print(f"  {true_pos / (true_pos + false_pos):.4f}")

**The lesson:** when the base rate is tiny, false positives from the huge healthy group
swamp the true positives. This is the *base rate fallacy*, and it is the reason a fraud
detector with "99% accuracy" can still be useless.

Let's see how the posterior depends on the prior.

In [ ]:
priors = np.linspace(0.0001, 0.5, 400)
posteriors = (sensitivity * priors) / (sensitivity * priors + (1 - specificity) * (1 - priors))

plt.plot(priors, posteriors, lw=2, color="steelblue")
plt.plot(priors, priors, ls="--", color="grey", label="no information (posterior = prior)")
plt.scatter([prior_disease], [posterior], color="crimson", zorder=5,
            label=f"our case: prior={prior_disease}, posterior={posterior:.3f}")
plt.xlabel("Prior P(disease) — how common the disease is")
plt.ylabel("Posterior P(disease | positive test)")
plt.title("The same test is convincing or useless depending on the base rate")
plt.legend()
plt.show()

### Bayes in code: a two-word naive Bayes spam filter

This is a preview of the Naive Bayes classifier you will meet in the ML module. The word
"naive" refers to the assumption that words appear **independently** given the class —
usually false, but often good enough.

In [ ]:
# Training statistics gathered from a labelled corpus
p_spam = 0.40
word_likelihood = pd.DataFrame({
    "word":            ["free", "meeting", "winner", "report"],
    "P(word | spam)":  [0.60,   0.02,      0.35,     0.03],
    "P(word | ham)":   [0.05,   0.30,      0.01,     0.40],
}).set_index("word")

def classify(words):
    '''Return P(spam | words) under the naive-independence assumption.'''
    lik_spam = np.prod([word_likelihood.loc[w, "P(word | spam)"] for w in words])
    lik_ham  = np.prod([word_likelihood.loc[w, "P(word | ham)"]  for w in words])
    num = lik_spam * p_spam
    return num / (num + lik_ham * (1 - p_spam))

for msg in (["free", "winner"], ["meeting", "report"], ["free", "report"]):
    print(f"{str(msg):30s} -> P(spam) = {classify(msg):.4f}")

---
## 1.8 Counting techniques

Classical probability is *"favourable ÷ total"*, so you need to count. Two tools cover
almost every case.

**Permutations — order matters.** Number of ways to arrange $r$ items out of $n$:

$$P(n, r) = \frac{n!}{(n-r)!}$$

**Combinations — order does not matter.** Number of ways to *choose* $r$ items from $n$:

$$C(n, r) = \binom{n}{r} = \frac{n!}{r!\,(n-r)!}$$

Ask yourself: *"if I swap two of the chosen items, is it a different result?"* Yes →
permutation. No → combination.

In [ ]:
n, r = 5, 3
print(f"Permutations P({n},{r}) = {perm(n, r)}")
print(f"Combinations C({n},{r}) = {comb(n, r)}")
print(f"Check: P = C * r!  ->  {comb(n, r)} * {factorial(r)} = {comb(n, r) * factorial(r)}\n")

items = ["A", "B", "C", "D", "E"]
print("Permutations of size 3 (first 8):", [''.join(p) for p in permutations(items, 3)][:8])
print("Combinations of size 3 (all)   :", [''.join(c) for c in combinations(items, 3)])

In [ ]:
# Poker: probability of being dealt a flush (5 cards of the same suit) from a 52-card deck
total_hands = comb(52, 5)
flush_hands = 4 * comb(13, 5)          # 4 suits, choose 5 of the 13 cards in that suit

print(f"Total 5-card hands : {total_hands:,}")
print(f"Flush hands        : {flush_hands:,}")
print(f"P(flush)           = {flush_hands / total_hands:.6f}  (about 1 in {total_hands // flush_hands})")

### The birthday problem

*How many people do you need in a room before there is a >50% chance that two share a
birthday?* Almost everyone guesses ~180. The real answer is **23**.

The trick is the complement again: compute $P(\text{all different})$ and subtract.

$$P(\text{all }n\text{ different}) = \frac{365}{365}\cdot\frac{364}{365}\cdots\frac{365-n+1}{365}$$

In [ ]:
def p_shared_birthday(n, days=365):
    p_all_different = np.prod([(days - i) / days for i in range(n)])
    return 1 - p_all_different

ns = np.arange(1, 81)
ps = [p_shared_birthday(n) for n in ns]

first_over_half = next(n for n in ns if p_shared_birthday(n) > 0.5)
print(f"Smallest group with P(shared birthday) > 0.5 : {first_over_half} people "
      f"({p_shared_birthday(first_over_half):.4f})")
print(f"With 50 people: {p_shared_birthday(50):.4f}")

plt.plot(ns, ps, lw=2, color="steelblue")
plt.axhline(0.5, color="crimson", ls="--")
plt.axvline(first_over_half, color="crimson", ls="--")
plt.xlabel("Number of people in the room")
plt.ylabel("P(at least two share a birthday)")
plt.title("The birthday problem")
plt.show()

In [ ]:
# Simulation sanity check for n = 23
n_people, n_trials = 23, 100_000
bdays = rng.integers(0, 365, size=(n_trials, n_people))
# a trial has a shared birthday if the number of unique values is less than n_people
shared = np.array([len(np.unique(row)) < n_people for row in bdays])
print(f"Theory     : {p_shared_birthday(23):.4f}")
print(f"Simulation : {shared.mean():.4f}")

---
## 1.9 Simulation as a safety net

You will not always be able to derive a probability on paper — but you can almost always
simulate it. The recipe:

1. Write a function that runs the experiment **once** and returns `True`/`False`
2. Run it many times (100,000 is usually plenty)
3. Take the mean

The **Monty Hall problem** is the perfect example. A game show has 3 doors; a car is
behind one, goats behind the other two. You pick a door. The host — who knows where the
car is — opens a *different* door revealing a goat, and offers you the chance to switch.
Should you?

In [ ]:
def monty_hall_once(switch, rng):
    doors = [0, 1, 2]
    car = rng.integers(0, 3)
    pick = rng.integers(0, 3)

    # Host opens a door that is neither your pick nor the car
    options = [d for d in doors if d != pick and d != car]
    opened = options[rng.integers(0, len(options))]

    if switch:
        pick = [d for d in doors if d != pick and d != opened][0]
    return pick == car

trials = 50_000
stay   = np.mean([monty_hall_once(False, rng) for _ in range(trials)])
switch = np.mean([monty_hall_once(True,  rng) for _ in range(trials)])

print(f"Win rate if you STAY   : {stay:.4f}   (theory 1/3 = 0.3333)")
print(f"Win rate if you SWITCH : {switch:.4f}   (theory 2/3 = 0.6667)")

**Why switching wins:** your first pick is right 1/3 of the time. The host's action never
changes that. So the remaining door absorbs the whole other 2/3. In Bayesian terms, the
host's choice is *informative* precisely because he is not choosing at random — he always
avoids the car.

---
## 1.10 Common mistakes to avoid

| Mistake | Why it's wrong |
|---|---|
| Confusing $P(A\mid B)$ with $P(B\mid A)$ | *P(positive \| sick)* = 0.99 but *P(sick \| positive)* = 0.02 |
| Assuming "mutually exclusive" means "independent" | Mutually exclusive events are maximally dependent |
| Multiplying probabilities of dependent events | Use $P(A\cap B) = P(A\mid B)P(B)$, not $P(A)P(B)$ |
| Ignoring the base rate | The disease example above |
| Adding overlapping events | Forgetting the $-P(A\cap B)$ term |
| Trusting a small sample | Law of Large Numbers needs *large* numbers |

---
## Exercises

Try each one **before** looking at the solution cell. Write code, not just algebra —
then check your algebra against the simulation.

**Exercise 1.** Two fair dice are rolled. Find $P(\text{sum is even} \mid \text{at least one die shows a 3})$.
Compute it by counting on the sample space, then verify by simulating 200,000 rolls.

In [ ]:
# --- Solution 1 -------------------------------------------------------------
U = set(S_two_dice)
B = {o for o in U if 3 in o}                 # at least one 3
A = {o for o in U if sum(o) % 2 == 0}        # sum is even

p_exact = len(A & B) / len(B)
print(f"Counting   : |A and B| / |B| = {len(A & B)}/{len(B)} = {p_exact:.4f}")

r = rng.integers(1, 7, size=(200_000, 2))
mask_B = (r == 3).any(axis=1)
p_sim = (r[mask_B].sum(axis=1) % 2 == 0).mean()
print(f"Simulation : {p_sim:.4f}")

**Exercise 2.** A factory's parts come from machines A (50%), B (30%), C (20%) with defect
rates 1%, 3%, 6%. A part is found defective. What is the probability it came from machine C?
(This is Bayes' theorem applied to the table in section 1.6.)

In [ ]:
# --- Solution 2 -------------------------------------------------------------
m = machines.copy()
m["posterior"] = m["contribution"] / m["contribution"].sum()
print(m[["machine", "share_of_output", "defect_rate", "posterior"]].to_string(index=False))
print(f"\nP(machine C | defective) = {m.loc[m.machine == 'C', 'posterior'].item():.4f}")
print("Machine C makes only 20% of parts but causes 40% of defects.")

**Exercise 3.** A quiz has 10 multiple-choice questions, 4 options each. A student guesses
every answer. Use counting to find the probability of getting **exactly 3** correct, then
confirm with simulation.

*Hint:* choose which 3 of the 10 are correct, then multiply by the probability of that
exact pattern: $\binom{10}{3}(0.25)^3(0.75)^7$.

In [ ]:
# --- Solution 3 -------------------------------------------------------------
p_correct = 0.25
k, n_q = 3, 10

p_theory = comb(n_q, k) * p_correct**k * (1 - p_correct)**(n_q - k)
print(f"Theory     : C(10,3) * 0.25^3 * 0.75^7 = {p_theory:.4f}")

guesses = rng.random((300_000, n_q)) < p_correct
print(f"Simulation : {(guesses.sum(axis=1) == k).mean():.4f}")
print("\n(You have just derived the Binomial distribution — see Notebook 3.)")

**Exercise 4 (challenge).** Three cards are in a hat: one red on both sides, one white on
both sides, one red on one side and white on the other. You draw a card at random and see
that the face-up side is **red**. What is the probability the other side is also red?

Most people say 1/2. Simulate it, then explain the answer with Bayes.

In [ ]:
# --- Solution 4 -------------------------------------------------------------
# Cards as (side1, side2)
cards = [("R", "R"), ("W", "W"), ("R", "W")]

trials, red_up, both_red = 200_000, 0, 0
for _ in range(trials):
    card = cards[rng.integers(0, 3)]
    up = rng.integers(0, 2)                    # which side is showing
    if card[up] == "R":
        red_up += 1
        if card[1 - up] == "R":
            both_red += 1

print(f"Simulation : P(other side red | this side red) = {both_red / red_up:.4f}")
print("Theory     : 2/3 = 0.6667")
print()
print("Why: there are 6 equally likely FACES, 3 of them red. Two of those red faces")
print("belong to the red/red card. Conditioning on 'a red face is showing' leaves 3")
print("possibilities, and 2 of them have red on the reverse.")
print("Counting FACES, not cards, is the key.")

---
## Summary

| Concept | Formula |
|---|---|
| Classical probability | $P(E) = \dfrac{|E|}{|S|}$ |
| Complement | $P(E^c) = 1 - P(E)$ |
| Addition rule | $P(A \cup B) = P(A) + P(B) - P(A \cap B)$ |
| Conditional probability | $P(A \mid B) = \dfrac{P(A \cap B)}{P(B)}$ |
| Multiplication rule | $P(A \cap B) = P(A \mid B)P(B)$ |
| Independence | $P(A \cap B) = P(A)P(B)$ |
| Total probability | $P(A) = \sum_i P(A \mid B_i)P(B_i)$ |
| Bayes' theorem | $P(B \mid A) = \dfrac{P(A \mid B)P(B)}{P(A)}$ |
| Permutations / Combinations | $P(n,r) = \dfrac{n!}{(n-r)!}$, $\;\binom{n}{r} = \dfrac{n!}{r!(n-r)!}$ |

**Next up:** [Notebook 2 — Random Variables](2.%20Random%20Variables.ipynb), where we
attach *numbers* to outcomes and start talking about expectation and variance.